<a href="https://colab.research.google.com/github/saraziadatt/bbb-exc-score/blob/main/colab_notebooks/bbb_exc_score_colab_V0_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Manually Install Libraries
# @markdown Restart runtime after installation.
# restarts runtime after first attempt
!pip install rdkit==2025.09.2
!pip install mordred==1.2.0

In [ ]:
# @title Import Packages
import pandas as pd
import numpy as np
import lightgbm as lgb
from datetime import datetime
import os

from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# @title Clone Repository
if not os.path.exists("/content/bbb-exc-score"):
    %cd /content
    !git clone https://github.com/saraziadatt/bbb-exc-score.git
    %cd /content/bbb-exc-score/final_model

#TODO add option to save output to google drive

In [ ]:
# @title User SMILES Input (Single Molecule)
#This is now redundant
SMILES = "" # @param {type:"string", placeholder:"paste SMILES here"}
mol_name = ""# @param {type:"string", placeholder:"name"}
mol = Chem.MolFromSmiles(SMILES)
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, AllChem.ETKDG())
AllChem.UFFOptimizeMolecule(mol)
mol.SetProp("_Name", mol_name)
writer = Chem.SDWriter("./user_single.sdf")
writer.write(mol)
writer.close()
print(mol_name, SMILES)

In [ ]:
# @title User SMILES Input (Multiple Molecules)
SMILES_space_separated = "" # @param {type:"string", placeholder:"paste SMILES here, separated by spaces"}
mol_names_optional = ""# @param {type:"string", placeholder:"paste molecule names (optional)"}

SMILES = SMILES_space_separated.split()
mol_names = mol_names_optional.split()

i=0
writer = Chem.SDWriter("./user_multi.sdf")
for smile in SMILES:
    try:
      mol_name = mol_names[i]
    except IndexError:
      mol_name = "NA"
    mol = Chem.MolFromSmiles(smile)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    AllChem.UFFOptimizeMolecule(mol)
    mol.SetProp("_Name", mol_name)
    writer.write(mol)
    i+=1
writer.close()

In [ ]:
# @title Read Input File
# @markdown ---
# @markdown For externally created 3D SDF files (V3000 format), upload as /content/bbb-exc-score/final_model/user_custom.sdf in file pane to the left.
# sdf must use V3000 format
user_file = 'test_final_model.sdf' # @param ["test_final_model.sdf", "user_single.sdf", "user_multi.sdf", "user_custom.sdf"]
data_warrior_3D_file = './'+ user_file # changed to input parameter

suppl = Chem.SDMolSupplier(data_warrior_3D_file)

SMILES = []
NAMES = []
for mol in suppl:
    if mol is None:   # skip malformed molecules
        continue
    smiles = Chem.MolToSmiles(mol)
    SMILES.append(smiles)
    NAMES.append(mol.GetProp('_Name')) # Added list of compound names

In [ ]:
# @title Calculate Descriptors
# Add all features

# Add ECFP Descriptors
fpgen = AllChem.GetMorganGenerator(radius=2)
ecfp_fingerprints = []
for smile in SMILES:
    mol = Chem.MolFromSmiles(smile)
    fp = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2).ToBitString()
    # print(list(fp))
    ecfp_fingerprints.append(list(fp))

ecfp_col_names = [f"ECFP_{i}" for i in range(0,2048)]
df = pd.DataFrame(ecfp_fingerprints, columns=ecfp_col_names)


# Add Mordred Descriptors
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
unique_mordred_filename = f'mordred_labelled_dataset_{timestamp}.csv'

cmd = f'python -m mordred -3 {data_warrior_3D_file} -o {unique_mordred_filename}'
os.system(cmd)

mordred_df = pd.read_csv(unique_mordred_filename)

# Add MACCS Descriptors
def get_maccs_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return list(MACCSkeys.GenMACCSKeys(mol).ToList())

col_names = [f"MACCS_{i}" for i in range(0,167)]
maccs_fps = []
for smile in SMILES:
    maccs_fps.append(get_maccs_fingerprint(smile))

maccs_fp_df = pd.DataFrame(data=maccs_fps, columns=col_names)

all_labelled_df = pd.concat([df, mordred_df, maccs_fp_df], axis=1)

# should probably create a pickle file of this to keep the correct columns.
cols_in_og = list(pd.read_csv(f'./bbbx_training_set.csv').columns)
cols_to_keep = [col for col in list(all_labelled_df.columns) if col in cols_in_og]

fully_labelled_df = all_labelled_df.loc[:, cols_to_keep]
fully_labelled_df.to_csv(f'./fully_labelled_dataset_{timestamp}.csv')
print(f'The fully labelled input dataset was saved to: ./fully_labelled_dataset_{timestamp}.csv', )
X_all = fully_labelled_df.values



In [ ]:
# @title Get Score Values
X_all = fully_labelled_df.values

all_predictions = []
for model_id in range(0,5):
    # TODO: Add the 15 values here
    model = lgb.Booster(model_file=f'./models/custom_mtl_original_outer_{model_id}.txt')
    predictions = model.predict(X_all)
    all_predictions.append(predictions)

all_preds = np.mean(np.array(all_predictions), axis=0)

scaler = MinMaxScaler()
scaler.fit(np.array([-2.7, 1, 1.7]).reshape(-1, 1))
y_pred_norm = scaler.transform(all_preds.reshape(-1, 1)).flatten()
final_scores = y_pred_norm*6

print(SMILES)

output_df_data = {'name': NAMES,'smiles': SMILES, 'predicted_logBB': all_preds, 'BBBX_score': final_scores} # Added names to output
output_df = pd.DataFrame(data=output_df_data)
unique_file_name = f'./predicted_scores_{timestamp}.csv'
output_df.to_csv(unique_file_name)

print(f'Predictions were saved to {unique_file_name}')

In [ ]:
# @title Display Output
output_df #changed so reading df not csv